# Effective dimensionality $d_{\rm eff}(x,\tau)$: September zfloor runs (5 seeds)

Copy of `effective_dimension_deff.ipynb` for the 5 seeds (900-904) of `zfloor_two_phase_reg1e-4`. Same estimators (`codes/effective_dimension.py`, derivation in `notes/effective_dimension_MGD.tex`): **denoiser form** (primary) and **energy form** (cross-check), as post-processing of each seed's $\Theta_{\rm reg}$, no SDE rerun.

What differs from the July notebook:
- $\Theta_{\rm reg}$ and its time grid come from the offline solve, `saved_results/theta_reg_lamsweep/<config>/lam<LAM>_ridge<RIDGE>.pt` (these runs used `--skip_reg_solve`); `t_reg` is saved there, so `reconstruct_t_reg` is not needed.
- That grid has ~60000 nodes (two_phase: 10000 on $[0,0.9)$, 50000 on $[0.9,1)$), so the evaluation uses `N_T_SUB` nodes **evenly spaced in $\log\tau$**, the same nodes for every seed (the grid is identical across seeds, checked).
- Data: n1 = 8500, 512 → 256 with one coarse-graining; potentials built with the runs' filter deduplication.
- The evaluation points use the same rare/typical split as `turb_theta_reg_rare_typical_sep.ipynb` (p95 of max |lag-1 increment|).

`LAM` sets which $\Theta_{\rm reg}$ is used. At `LAM = 0` (no time smoothing) every node is its own noisy solve, and the $t\to1$ end has very large coefficients (seen in `lamtune_select_zfloor.ipynb`); the small-$\tau$ end of every curve is the least trustworthy.

In [ ]:
import sys, os, re, math, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

root = Path().resolve()          # turbulence/
sys.path.insert(0, str(root / '../codes'))
sys.path.insert(0, str(root / '../data'))

from filters_bank import return_Filters
from potentials_builder import get_1d_potentials
from utils import split_periodize_reshape, normalize
from ortho_wavelet.ReadyToUseWavelets import DefineWavelet
from effective_dimension import (
    load_fitted_potentials, load_or_recompute_m_t,
    h_bound_at, extract_rare_typical, denoiser_d_eff, energy_d_eff,
    alpha_of_t, tau_of_t,
)
from data_loader import load_turbulence_1d

RUN_LOCATION = "jean_zay"   # "local" or "jean_zay"
if RUN_LOCATION == "local":
    local_turbulence_path = (root / "../data/data_files/turbulence_1d_period.pt").resolve()
    if local_turbulence_path.exists():
        os.environ["TURBULENCE_1D_DATA_PATH"] = str(local_turbulence_path)
    else:
        print(f"WARNING: {local_turbulence_path} not found -- load_turbulence_1d() "
              f"will fall back to its Jean Zay default path and fail locally.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', root)

## Runs, `LAM`, and the SMOKE_TEST toggle

In [ ]:
GROUP = 'turbulencesynth_M256_J8_Q3_sigma3.5_nt60000_n1_8500_lam0.0_thomas_nsub1_f64_dedupfilt1e-08_terms2f1130a8_zfloor_two_phase_reg1e-4'
RUN = ('turbulencesynth_M256_J8_Q3_sigma3.5_nt60000_n1_8500_lam0.0_thomas_nsub1_f64_dedupfilt1e-08'
       '_seed_{seed}_terms2f1130a8_zfloor_two_phase_reg1e-4_{ts}')
CONFIGS = [RUN.format(seed=900, ts='20260928_1545'),                       # SLURM job 272923
           *(RUN.format(seed=s, ts='20260928_1644') for s in (901, 902, 903, 904))]   # array job 276063
LAM = 0.0           # lam of the offline Theta_reg solve (0 = unsmoothed, as in the rare/typical run)
RIDGE = 1e-6        # as in launch/resolve_lamsweep.sh

# ----------------------------------------------------------------------------
SMOKE_TEST = False   # True: tiny sizes, only checks the pipeline runs (NOT a result)
# ----------------------------------------------------------------------------
if SMOKE_TEST:
    N_SEEDS = 2; N_T_SUB = 12; N_MC = 8; N_PER_CATEGORY = 10; M1_RECOMPUTE_SUBSET = 300
else:
    N_SEEDS = None            # None = all of CONFIGS
    N_T_SUB = 200             # nodes of the ~60000-node t_reg grid, evenly spaced in log(tau)
    N_MC = 128
    N_PER_CATEGORY = 100
    M1_RECOMPUTE_SUBSET = 2000
print(f"SMOKE_TEST={SMOKE_TEST}  N_SEEDS={N_SEEDS} N_T_SUB={N_T_SUB} N_MC={N_MC} N_PER_CATEGORY={N_PER_CATEGORY}")

base = root / 'saved_results'
sweep_root = base / 'theta_reg_lamsweep'

def load_pt(path):
    path = Path(path)
    p_pt = path.with_name(path.name + '.pt')
    return torch.load(p_pt if p_pt.exists() else path, map_location='cpu', weights_only=False)

exp_dir_by_seed, cfg_by_seed, config_by_seed = {}, {}, {}
for config in CONFIGS[:N_SEEDS]:
    exp_dir = root / 'experiments' / GROUP / config
    assert exp_dir.is_dir(), f'missing {exp_dir}'
    c = json.loads((exp_dir / 'config.json').read_text())
    key = c['seed']
    exp_dir_by_seed[key], cfg_by_seed[key], config_by_seed[key] = exp_dir, c, config
    path = sweep_root / config / f'lam{LAM:g}_ridge{RIDGE:g}.pt'
    assert path.exists(), f'not solved: {path}'
seeds = sorted(cfg_by_seed)
cfg = cfg_by_seed[seeds[0]]
skip = {'timestamp', 'config_name', 'seed'}
for s in seeds[1:]:
    diff = {q for q in set(cfg) | set(cfg_by_seed[s]) if q not in skip and cfg.get(q) != cfg_by_seed[s].get(q)}
    assert not diff, f'seed {s} differs from seed {seeds[0]} in {diff}'
print(f"{len(seeds)} seeds {seeds}, identical config.json apart from seed/timestamp")

J, Q, M = cfg['J'], cfg['Q'], cfg['M']
n1, terms = cfg['n1'], cfg['terms']

## Data (shared across all seeds: these runs all fit the same `x1`)

In [ ]:
W = DefineWavelet('Db', m=3, device=device)
Data_raw = load_turbulence_1d().to(device)
Data = split_periodize_reshape(Data_raw, cfg['subseries_len'])
for _ in range(int(np.log2(cfg['subseries_len'] / cfg['target_len']))):   # as run_SDE.py
    Data = W.decompose(Data)[1]
x1 = normalize(Data[:n1]).to(device)
d = int(np.prod(x1.shape[1:]))
assert d == M
print('Data:', tuple(Data.shape), '| x1:', tuple(x1.shape), '| d =', d)

## Rare / typical split and fixed evaluation points

Same criterion as `turb_theta_reg_rare_typical_sep.ipynb` (p95 of max |lag-1 increment|; `normalize` is global, so splitting `Data[:n1]` or `x1` gives the same indices). The SAME points are evaluated for every seed (fixed RNG).

In [ ]:
idx_typical, idx_rare, split_info = extract_rare_typical(Data[:n1], criterion='percentile', percentile=95)
print(f"typical {len(idx_typical)}, rare {len(idx_rare)}   {split_info}")

rng = np.random.default_rng(0)
picked_typical = rng.choice(idx_typical, size=N_PER_CATEGORY, replace=False)
picked_rare = rng.choice(idx_rare, size=N_PER_CATEGORY, replace=False)
eval_idx = np.concatenate([picked_typical, picked_rare])
eval_category = ['typical'] * N_PER_CATEGORY + ['rare'] * N_PER_CATEGORY
x_eval = x1[eval_idx]
print('x_eval:', tuple(x_eval.shape))

## Theta_reg, time grid and fitted potentials per seed

The potentials are built as `run_experiment` builds them (terms order, filter deduplication), then each seed's fitted state is loaded. `load_fitted_potentials` silently keeps an unfitted potential if its state file is missing, so the files are checked here, and $r$ is checked against $\Theta_{\rm reg}$'s width.

In [ ]:
filters, filters_Phi = return_Filters(M, J, 1, device=device, include_phi=True)
filters_Q = return_Filters(M, J, Q, device=device)
potentials_skeleton = get_1d_potentials(
    terms, J, filters, Q, filters_Q=filters_Q, filters_Phi=filters_Phi,
    scalar_param=None, parallel=False,
    deduplicate_filters=cfg.get('deduplicate_filters', False), dedup_tol=cfg.get('dedup_tol', 1e-12),
)

Theta_by_seed, potentials_by_seed = {}, {}
t_reg_full = None
for s in seeds:
    sol = load_pt(sweep_root / config_by_seed[s] / f'lam{LAM:g}_ridge{RIDGE:g}.pt')
    assert sol['config'] == config_by_seed[s]
    Theta_by_seed[s] = sol['Theta_reg'].double()
    t_s = sol['t_reg'].double().numpy()
    assert len(t_s) == Theta_by_seed[s].shape[0]
    if t_reg_full is None:
        t_reg_full = t_s
    else:
        assert len(t_s) == len(t_reg_full) and np.allclose(t_s, t_reg_full), f'seed {s}: t_reg differs'

    pdir = exp_dir_by_seed[s] / 'fitted_potentials'
    missing = [n for n, p in potentials_skeleton.items() if hasattr(p, 'is_fitted') and not (pdir / f'{n}.pt').exists()]
    assert not missing, f'seed {s}: no fitted state for {missing} in {pdir}'
    potentials_by_seed[s] = load_fitted_potentials(potentials_skeleton, pdir, device=device)
    r = sum(p.num_coefficients for p in potentials_by_seed[s].values())
    assert r == Theta_by_seed[s].shape[1], f'seed {s}: r = {r} but Theta_reg has {Theta_by_seed[s].shape[1]} columns'
    print(f"seed {s}: Theta_reg {tuple(Theta_by_seed[s].shape)}, lam={sol['lam']:g}, residual {sol['residual']:.2e}, r={r}")

# Evaluation nodes: N_T_SUB targets evenly spaced in log(tau), nearest grid node each
tau_full = tau_of_t(t_reg_full)
if N_T_SUB is not None and N_T_SUB < len(t_reg_full):
    targets = np.linspace(np.log(tau_full).max(), np.log(tau_full).min(), N_T_SUB)
    sub = np.unique([int(np.argmin(np.abs(np.log(tau_full) - g))) for g in targets])
else:
    sub = np.arange(len(t_reg_full))
t_reg = t_reg_full[sub]
print(f"{len(sub)} evaluation nodes of {len(t_reg_full)}: t in [{t_reg[0]:.3g}, {t_reg[-1]:.8f}], "
      f"tau in [{tau_of_t(t_reg).min():.2e}, {tau_of_t(t_reg).max():.2e}], "
      f"{int((t_reg >= 0.9).sum())} of them in t >= 0.9")

## Per-seed computation

For each seed: $m_t$ from the run's `aux_moments` (recomputed from `x1` if absent), $H_{\rm bound}(t)$ from the saved `dH_t_bound` on the run's own fine grid (`sampling_times`), then `denoiser_d_eff` and `energy_d_eff` at the fixed evaluation points and nodes.

In [ ]:
rows = []
t0 = time.time()
for s in seeds:
    t_seed0 = time.time()
    config = config_by_seed[s]
    potentials = potentials_by_seed[s]
    theta_reg = Theta_by_seed[s][sub]
    t_fine = load_pt(base / 'sampling_times' / config).numpy().astype(np.float64)
    dH_fine = load_pt(base / 'entropy_bounds' / config).numpy().astype(np.float64)

    aux_path = base / 'aux_moments' / f'{config}_aux_moments.pt'
    m_t, m_t_source = load_or_recompute_m_t(aux_path, t_fine, t_reg, potentials, x1[:M1_RECOMPUTE_SUBSET], n_mc_noise=1)
    H_bound_t_reg = h_bound_at(t_fine, dH_fine, t_reg, d=d)

    gen = torch.Generator(device=x_eval.device).manual_seed(s)
    d_den = denoiser_d_eff(x_eval, theta_reg, t_reg, potentials, n_mc=N_MC, generator=gen)
    d_en = energy_d_eff(x_eval, theta_reg, t_reg, potentials, m_t, H_bound_t_reg, d=d, n_mc=N_MC, generator=gen)

    tau_reg = tau_of_t(t_reg)
    for j in range(len(t_reg)):
        for k in range(x_eval.shape[0]):
            rows.append(dict(seed=s, m_t_source=m_t_source, point_idx=int(eval_idx[k]), category=eval_category[k],
                             t=float(t_reg[j]), tau=float(tau_reg[j]),
                             d_eff_denoiser=float(d_den[j, k]), d_eff_energy=float(d_en[j, k])))
    print(f"seed {s}: {len(t_reg)} t-nodes x {x_eval.shape[0]} points, m_t={m_t_source}  "
          f"[{time.time()-t_seed0:.1f}s, total {time.time()-t0:.1f}s]")

results_df = pd.DataFrame(rows)
print(f"\nTotal: {len(results_df)} rows, {results_df['seed'].nunique()} seeds, {time.time()-t0:.1f}s")
results_df.head()

## Save the aggregate results (under `deff_results/`, not `saved_results/`)

In [ ]:
out_dir = root / 'deff_results'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'd_eff_zfloor_two_phase_lam{LAM:g}_smoke{SMOKE_TEST}_nseeds{results_df["seed"].nunique()}.csv'
results_df.to_csv(out_path, index=False)
print('saved to', out_path)

## Per-seed curves, typical vs rare

Thin lines: one (seed, point); thick line: median over the category at each $\tau$ node. $x$-axis is $\tau$ (log scale, inverted so $t$ increases to the right). The right panel repeats the left with a clipped $y$-range.

In [ ]:
colors = {'typical': 'tab:blue', 'rare': 'tab:red'}

def plot_curves(ylim=None):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, col, title in [(axes[0], 'd_eff_denoiser', 'denoiser form'),
                           (axes[1], 'd_eff_energy', 'energy form')]:
        for cat in ['typical', 'rare']:
            sub_c = results_df[results_df.category == cat]
            for _, g in sub_c.groupby(['seed', 'point_idx']):
                g = g.sort_values('tau')
                ax.plot(g['tau'], g[col], color=colors[cat], alpha=0.02, lw=0.8)
            med = sub_c.groupby('tau')[col].median().sort_index()
            ax.plot(med.index, med.values, color=colors[cat], lw=1.5, label=f'{cat} (median)')
        ax.axhline(d, color='k', ls='--', lw=1, label=f'd = {d}')
        ax.set_xscale('log'); ax.invert_xaxis()
        ax.set_xlabel(r'$\tau$'); ax.set_title(title); ax.grid(alpha=0.3)
    axes[0].set_ylabel(r'$d_{\rm eff}(x,\tau)$'); axes[0].legend(fontsize=8)
    if ylim is not None:
        axes[0].set_ylim(*ylim)
    fig.suptitle(f'Effective dimensionality, {results_df["seed"].nunique()} seed(s), lam={LAM:g}, '
                 f'{N_PER_CATEGORY} typical + {N_PER_CATEGORY} rare points'
                 + (' -- SMOKE TEST, not a real result' if SMOKE_TEST else ''))
    fig.tight_layout(); plt.show()

plot_curves()
plot_curves(ylim=(-0.5 * d, 2.5 * d))

## Summary: typical vs rare, mean $\pm$ std across seeds

Average over the evaluation points of a category first, then mean and std **across seeds** at each $\tau$ node. The shaded grey band marks the $\tau$ nodes where the all-point mean denoiser $d_{\rm eff}$ exceeds $d$ (outside the interpretable window).

In [ ]:
mean_all = results_df.groupby('tau').d_eff_denoiser.mean().sort_index()
tau_over_d = mean_all.index[mean_all.values > d].values
print(f"tau nodes with mean denoiser d_eff > d: {len(tau_over_d)} of {len(mean_all)}"
      + (f", tau in [{tau_over_d.min():.2e}, {tau_over_d.max():.2e}]" if len(tau_over_d) else ''))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, col, title in [(axes[0], 'd_eff_denoiser', 'denoiser form'),
                       (axes[1], 'd_eff_energy', 'energy form')]:
    for cat in ['typical', 'rare']:
        per_seed = results_df[results_df.category == cat].groupby(['seed', 'tau'])[col].mean().reset_index()
        st = per_seed.groupby('tau')[col].agg(['mean', 'std']).sort_index()
        ax.plot(st.index, st['mean'], color=colors[cat], lw=2, label=cat)
        ax.fill_between(st.index, st['mean'] - st['std'], st['mean'] + st['std'], color=colors[cat], alpha=0.2)
    ax.axhline(d, color='k', ls='--', lw=1, label=f'd = {d}')
    for tt in tau_over_d:
        ax.axvline(tt, color='0.85', lw=2, zorder=0)
    ax.set_xscale('log'); ax.invert_xaxis()
    ax.set_xlabel(r'$\tau$'); ax.set_title(title); ax.grid(alpha=0.3)
axes[0].set_ylabel(r'$d_{\rm eff}(x,\tau)$'); axes[0].legend(fontsize=8)
fig.suptitle(f'Typical vs rare, mean ± std across seeds, lam={LAM:g}'
             + (' -- SMOKE TEST, not a real result' if SMOKE_TEST else ''))
fig.tight_layout(); plt.show()

## Caveats (see `notes/effective_dimension_MGD.tex` for the full derivation)

- **Denoiser form is the primary estimator.** It needs only $\Theta_{\rm
  reg}$ and $\nabla\varphi$; no $\log Z_t$, no $m_t$, no normalisation
  constant enters (it cancels analytically).
- **Energy form is a secondary cross-check, not an independent estimate.**
  `energy_d_eff` differentiates $\log Z_t \approx H_{\rm bound}(t) +
  \theta_t^\top m_t$ (MGD Eq. 8/30) -- a *lower bound* on $\log Z_t$, not
  $\log Z_t$ itself -- with a finite difference on a Monte-Carlo-noisy
  curve, so it inherits both the bound's bias and extra numerical noise
  from the derivative. Expect it to be visibly noisier than the denoiser
  form at the same `N_MC`; increase `N_MC` and/or `N_T_SUB` before trusting
  it quantitatively.
- **The last $t_{\rm reg}$ nodes ($\tau\to0$) use the most
  ill-conditioned $\theta$** (for these runs, coefficient 0 reaches about $\pm10^8$ near $t\to1$ at $\lambda\le10^{-8}$, see `lamtune_select_zfloor.ipynb`) -- treat the
  smallest-$\tau$ end of every curve as the least numerically trustworthy,
  not the most informative, until cross-checked against a finer grid /
  larger `N_MC`.
- `m_t_source` in `results_df` records whether $m_t$ came from
  `aux_moments` (pulled from Jean Zay) or was recomputed locally from
  `x1` -- both should agree up to Monte Carlo noise; worth checking once
  `aux_moments` is available.

## Continuous version: $d_{\rm eff}$ vs $\log p_\theta$ and extremeness

At fixed τ (`TAU_TARGETS`, pick them inside the interpretable window from the curves above: check the `frac d_eff > d` column), $d_{\rm eff}(x,\tau)$ on a few hundred points, rank-correlated (Spearman) with $\log p_\theta(x) = \Theta_{\rm reg}[-1]^\top\varphi(x) - \log Z$ ($p\propto e^{+\theta^\top\varphi}$) and with extremeness $e(x)=\max_u|x(u+1)-x(u)|$. Points: half uniform over `x1`, half from the top 5% in $e$ (tail oversampled, so the correlations describe this design, not the population). Error bars: bootstrap over points and std across seeds. Partial rank correlation $\rho(d_{\rm eff}, \log p \mid e)$ removes the part explained by $e$.

$\varphi$ is evaluated in float32 and cast to double before the product with $\Theta_{\rm reg}[-1]$, as in the July notebook.

In [ ]:
TAU_TARGETS = [1e-1, 1e-3]   # check against the curves above
N_POINTS = 40 if SMOKE_TEST else 400
N_MC_CONT = 8 if SMOKE_TEST else 128
POINT_CHUNK = 100            # points per call (memory)

e_all = (x1[..., 1:] - x1[..., :-1]).abs().amax(dim=(1, 2)).cpu().numpy()  # x1 units

rng_c = np.random.default_rng(1)
top = np.where(e_all > np.percentile(e_all, 95))[0]
pts = np.unique(np.concatenate([rng_c.choice(n1, N_POINTS // 2, replace=False),
                                rng_c.choice(top, N_POINTS // 2, replace=False)]))
x_pts = x1[pts]
is_rare_pts = np.isin(pts, idx_rare)
print(f"{len(pts)} points ({is_rare_pts.sum()} in idx_rare), e in [{e_all[pts].min():.3g}, {e_all[pts].max():.3g}]")

In [ ]:
to_np = lambda a: a.detach().cpu().numpy() if torch.is_tensor(a) else np.asarray(a)

def phi_of(potentials, x):
    with torch.no_grad():
        f = [p(x) for p in potentials.values()]
    return torch.cat([g.unsqueeze(1) if g.ndim == 1 else g for g in f], dim=-1)

nodes = [int(np.argmin(np.abs(np.log(tau_full) - np.log(tt)))) for tt in TAU_TARGETS]
print('tau nodes:', [f'{tau_full[n]:.2e}' for n in nodes])
chunks = [slice(i, i + POINT_CHUNK) for i in range(0, len(pts), POINT_CHUNK)]
rows_c = []
for s in seeds:
    potentials = potentials_by_seed[s]
    Theta_reg = Theta_by_seed[s]
    phi_pts = torch.cat([phi_of(potentials, x_pts[c]) for c in chunks]).cpu().double()
    assert phi_pts.shape[1] == Theta_reg.shape[1]
    logp = (phi_pts @ Theta_reg[-1]).numpy()          # log p_theta + log Z, same constant for all x

    gen = torch.Generator(device=x_pts.device).manual_seed(s)
    d_den = np.concatenate([to_np(denoiser_d_eff(x_pts[c], Theta_reg[nodes], t_reg_full[nodes],
                                                 potentials, n_mc=N_MC_CONT, generator=gen))
                            for c in chunks], axis=1)  # (n_nodes, n_points)
    for j, node in enumerate(nodes):
        rows_c += [dict(seed=s, tau_target=TAU_TARGETS[j], tau=float(tau_full[node]),
                        point_idx=int(pts[k]), d_eff=float(d_den[j, k]), logp=float(logp[k]),
                        e=float(e_all[pts[k]]), rare=bool(is_rare_pts[k])) for k in range(len(pts))]
    print(f"seed {s} done")
cont_df = pd.DataFrame(rows_c)

In [ ]:
def spearman(a, b):
    return np.corrcoef(pd.Series(np.asarray(a)).rank(), pd.Series(np.asarray(b)).rank())[0, 1]

def partial_spearman(a, b, c):
    ra, rb, rc = (pd.Series(np.asarray(v)).rank().values for v in (a, b, c))
    res = lambda y: y - np.polyval(np.polyfit(rc, y, 1), rc)
    return np.corrcoef(res(ra), res(rb))[0, 1]

per_point = (cont_df.groupby(['tau_target', 'point_idx'])
             .agg(d_eff=('d_eff', 'mean'), logp=('logp', 'mean'), e=('e', 'first'), rare=('rare', 'first'))
             .reset_index())

rows_s = []
for tt, gp in per_point.groupby('tau_target'):
    seed_rho = cont_df[cont_df.tau_target == tt].groupby('seed').apply(
        lambda q: pd.Series({'lp': spearman(q.d_eff, q.logp), 'e': spearman(q.d_eff, q.e)}))
    D, L, E = gp.d_eff.values, gp.logp.values, gp.e.values
    boot = np.array([[spearman(D[b], L[b]), spearman(D[b], E[b]), partial_spearman(D[b], L[b], E[b])]
                     for b in (rng_c.integers(0, len(gp), len(gp)) for _ in range(500))])
    rows_s.append({'tau': tt, 'median d_eff': np.median(D), 'frac d_eff > d': (D > d).mean(),
                   'rho(d_eff, log p)': spearman(D, L), 'SE boot': boot[:, 0].std(ddof=1),
                   'std seeds': seed_rho.lp.std(ddof=1),
                   'rho(d_eff, e)': spearman(D, E), 'SE boot (e)': boot[:, 1].std(ddof=1),
                   'std seeds (e)': seed_rho.e.std(ddof=1),
                   'rho(d_eff, log p | e)': partial_spearman(D, L, E), 'SE boot (partial)': boot[:, 2].std(ddof=1)})
display(pd.DataFrame(rows_s))

## Consistency check: $\log p_\theta$ vs the integrated deficit $\int (d - d_{\rm eff})\,ds$

Per (seed, evaluation point), the denoiser deficit integrated over $s=\tfrac12\log\tau$ down to $\tau_{\min}$, against $\log p_\theta = \Theta_{\rm reg}[-1]^\top\varphi$, both centered per seed (removes $\log Z$). Slope 1 is the target.

In [ ]:
TAU_MIN_LIST = [1e-2, 1e-3, 1e-4]   # lower cutoffs; keep them above the d_eff > d region
trap = getattr(np, 'trapezoid', None) or np.trapz

logp_rows = []
for s in seeds:
    lp = (phi_of(potentials_by_seed[s], x_eval).cpu().double() @ Theta_by_seed[s][-1]).numpy()   # log p_theta + log Z
    logp_rows += [dict(seed=s, point_idx=int(eval_idx[k]), logp=lp[k]) for k in range(len(eval_idx))]
logp_df = pd.DataFrame(logp_rows)

def deficit_integral(g, tau_min):
    g = g[g.tau >= tau_min].sort_values('tau')
    return trap(d - g.d_eff_denoiser.values, 0.5 * np.log(g.tau.values))

rows = []
for (s, pidx), g in results_df.groupby(['seed', 'point_idx']):
    rows.append({'seed': s, 'point_idx': pidx, 'category': g.category.iloc[0],
                 **{f'I_{tm:.0e}': deficit_integral(g, tm) for tm in TAU_MIN_LIST}})
check_df = pd.DataFrame(rows).merge(logp_df, on=['seed', 'point_idx'])

fig, axes = plt.subplots(1, len(TAU_MIN_LIST), figsize=(4.5 * len(TAU_MIN_LIST), 4), sharey=True)
yc = check_df.logp - check_df.groupby('seed').logp.transform('mean')   # remove per-seed constant
for ax, tm in zip(axes, TAU_MIN_LIST):
    col = f'I_{tm:.0e}'
    xc = check_df[col] - check_df.groupby('seed')[col].transform('mean')
    for cat, c in colors.items():
        m = check_df.category == cat
        ax.scatter(xc[m], yc[m], s=12, color=c, label=cat)
    lim = [min(xc.min(), yc.min()), max(xc.max(), yc.max())]
    ax.plot(lim, lim, 'k--', lw=1, label='slope 1')
    ax.set_title(rf'$\tau_{{\min}}$={tm:.0e}: slope {np.polyfit(xc, yc, 1)[0]:.2f}, r={np.corrcoef(xc, yc)[0, 1]:.2f}')
    ax.set_xlabel(r'$\int (d-d_{\rm eff})\,ds$ (centered per seed)')
axes[0].set_ylabel(r'$\log p_\theta$ (centered per seed)'); axes[0].legend(fontsize=8)
fig.tight_layout(); plt.show()

In [ ]:
mean_cat = results_df.groupby(['category', 'tau']).d_eff_denoiser.mean().unstack(0).sort_index()
tau_g = mean_cat.index.values
s_g = 0.5 * np.log(tau_g)
gap = (mean_cat['rare'] - mean_cat['typical']).values                  # Δd_eff(s), rare − typical
seg = 0.5 * (gap[1:] + gap[:-1]) * np.diff(s_g)
cum = np.concatenate([np.cumsum(seg[::-1])[::-1], [0.0]])              # ∫_s^{s_max} Δd_eff ds'

obs = logp_df.merge(check_df[['seed', 'point_idx', 'category']]).groupby(['seed', 'category']).logp.mean().unstack()
obs_gap = (obs['typical'] - obs['rare']).mean()                        # same points, mean over seeds

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tau_g, cum, color='k')
ax.axhline(obs_gap, color='tab:red', ls='--', label=f'observed log p gap, same points ({obs_gap:.3g})')
for tt in tau_over_d:
    ax.axvline(tt, color='0.85', lw=2, zorder=0)
ax.set_xscale('log'); ax.invert_xaxis()
ax.set_xlabel(r'$\tau$ (integration runs from large $\tau$ down to this value)')
ax.set_ylabel(r'$\int \Delta d_{\rm eff}\,ds$ (nats)')
ax.set_title(r'Cumulative contribution to $\log p_{\rm typ}-\log p_{\rm rare}$ (grey: mean $d_{\rm eff}>d$)')
ax.legend(fontsize=8); plt.show()